# 실습 5 손으로 쓴 숫자 분류 모델 구현하기 (교사용)

교과서 137~140쪽

교과서 실습 5와 같은 순서로 손글씨 숫자 데이터(MNIST)를 분류하는 신경망을 만들고 성능을 평가합니다. [단계 1] 문제 정의: 손으로 쓴 숫자 분류 모델을 구현해 보자.

**사용 방법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶ 버튼을 눌러 실행합니다. 코드는 접혀 있고, 궁금하면 칸을 두 번 눌러 펼칠 수 있습니다.
- 오른쪽의 슬라이더와 선택 상자로 설정을 바꿉니다. **설정을 바꾸면 ③과 ④를 다시 실행**하세요.
- 런타임은 기본(CPU)으로 충분합니다.

| 단계 | 할 일 |
|---|---|
| 기본 | ①~⑥을 기본 설정 그대로 실행하고 테스트 정확도를 기록한다 |
| 조작 | ③, ④의 설정을 바꿔 2~3번 더 실행하고 가장 좋은 설정을 찾는다 |
| 도전 | 맨 아래 도전 칸에서 코드 한 줄을 고쳐 층을 추가한다 |


> **[교사용]** 학생용과 코드는 같고, 칸마다 지도 메모가 붙어 있습니다. 수업 전에 한 번 끝까지 실행해 두면 첫 실행 시 데이터를 내려받는 시간(10초 안팎)과 학습 시간(CPU에서 한 번에 약 5~10초)을 미리 확인할 수 있습니다.

In [ ]:
#@title ① [단계 2] 데이터 수집 및 전처리하기
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 손글씨 숫자 7만 장: 훈련용 6만 장, 테스트용 1만 장
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# 픽셀값 0~255를 0~1로 바꾼다 (정규화)
X_train = X_train / 255.0
X_test = X_test / 255.0

print('훈련 데이터:', X_train.shape, '/ 테스트 데이터:', X_test.shape)
print('그림 한 장의 크기: 28 x 28 =', 28 * 28, '개의 픽셀')

> **[교사용]** `load_data()`는 처음 실행할 때만 데이터를 내려받습니다. 학생에게 "그림 한 장 = 784개의 수"를 꼭 짚어 줍니다. 교과서 그리기 판의 [숫자로 보기]와 연결됩니다.
> 255로 나누는 이유: 입력값의 범위를 0~1로 맞추면 학습이 안정적입니다. 01 중단원에서 배운 정규화와 같은 개념입니다.

In [ ]:
#@title ② 데이터 살펴보기: 번호를 바꿔 여러 숫자를 보세요 (교과서 138쪽)
sample_index = 0 #@param {type:"slider", min:0, max:59999, step:1}

plt.figure(figsize=(3, 3))
plt.imshow(X_train[sample_index], cmap='gray')
plt.title(f'label = {y_train[sample_index]}')
plt.axis('off')
plt.show()
print('이 그림의 정답(레이블):', y_train[sample_index])
print('가운데 줄(14번째 줄) 픽셀값 일부:', np.round(X_train[sample_index][14][6:22], 1))

> **[교사용]** 레이블은 사람이 붙인 정답입니다. 지도학습이라는 점을 다시 확인합니다. 슬라이더로 여러 장을 보며 "사람도 헷갈리는 숫자"를 찾게 하면 ⑥과 연결됩니다.

### 예측해 보기
기본 설정(은닉층 128개, 64개 / 학습 10번, 교과서와 같음)으로 학습하면 **테스트 정확도**가 몇 %쯤 나올까요? 실행하기 전에 연구 노트에 예측값을 적어 두세요.

In [ ]:
#@title ③ [단계 3] 신경망 모델 설계하기, 환경 설정하기 (교과서 139쪽)
hidden1 = 128 #@param [16, 32, 64, 128, 256] {type:"raw"}
hidden2 = 64 #@param [0, 16, 32, 64, 128] {type:"raw"}
activation = "relu" #@param ["relu", "sigmoid"]

def build_model():
    keras.utils.set_random_seed(42)            # 실행할 때마다 결과가 크게 달라지지 않게 고정
    stack = [keras.Input(shape=(28, 28)),
             layers.Flatten()]                  # 28x28 그림을 784개의 수로 펼친다 (입력층)
    stack.append(layers.Dense(hidden1, activation=activation))      # 은닉층 1
    if hidden2 > 0:
        stack.append(layers.Dense(hidden2, activation=activation))  # 은닉층 2 (0이면 없음)
    stack.append(layers.Dense(10, activation='softmax'))            # 출력층: 0~9일 확률

    model = keras.Sequential(stack)              # 1. 설계
    model.compile(loss='sparse_categorical_crossentropy',  # 2. 환경 설정: 손실 함수
                  optimizer='adam',                         #    최적화 함수
                  metrics=['accuracy'])                     #    평가 지표
    return model

model = build_model()
model.summary()
print('은닉층 설정:', hidden1, '/', hidden2 if hidden2 > 0 else '없음', '/ 활성화 함수:', activation)

> **[교사용]** `model.summary()`에서 파라미터 수를 함께 봅니다. 기본 설정은 784x128+128 + 128x64+64 + 64x10+10 = 109,386개입니다. 이 수가 모두 학습으로 정해지는 가중치와 편향이라는 점을 강조합니다.
> 손실 함수는 '얼마나 틀렸는지 재는 자', 최적화 함수(adam)는 '그 오차를 줄이는 방향으로 가중치를 고치는 방법'으로 설명하면 충분합니다.

In [ ]:
#@title ④ [단계 3] 신경망 모델 학습하기 (설정을 바꿨다면 ③부터 다시 실행)
epochs = 10 #@param {type:"slider", min:1, max:20, step:1}

model = build_model()   # 매번 처음 상태의 신경망으로 다시 학습한다
history = model.fit(X_train, y_train,          # 3. 학습
                    epochs=epochs,
                    validation_split=0.1,      # 훈련 데이터의 10%는 검증용으로 떼어 둔다
                    batch_size=128,
                    verbose=2)

plt.figure(figsize=(6, 3.5))
xs = range(1, epochs + 1)
plt.plot(xs, history.history['accuracy'], 'o-', label='train')
plt.plot(xs, history.history['val_accuracy'], 's-', label='validation')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

> **[교사용]** **예상 결과(대략, 실행마다 조금 다름)**: 교과서는 10회 학습 후 테스트 정확도 약 97.8%를 예시로 든다(140쪽). 기본 설정이면 비슷하게 나온다.
> - 시간이 부족하면 학습 횟수를 5로 줄여도 97% 안팎이 나온다.
> - 학습 횟수를 15~20으로 늘리면 훈련 정확도는 99%를 넘지만 검증 정확도는 거의 오르지 않거나 들쭉날쭉합니다. **과적합**을 설명하기 좋은 장면입니다.
> - 활성화 함수를 sigmoid로 바꾸면 같은 횟수에서 정확도가 조금 낮게 시작합니다. ReLU가 깊은 신경망에서 많이 쓰이는 이유로 연결할 수 있습니다.
> - 은닉층 1을 16으로 줄이면 정확도가 눈에 띄게 떨어집니다(대략 94~95%).

In [ ]:
#@title ⑤ [단계 4] 테스트 데이터로 모델 평가하기 (교과서 140쪽)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)   # 4. 평가
train_acc = history.history['accuracy'][-1]
print(f'훈련 정확도: {train_acc * 100:.2f}%')
print(f'테스트 정확도: {test_acc * 100:.2f}%')

pred = model.predict(X_test, verbose=0).argmax(axis=1)
cm = tf.math.confusion_matrix(y_test, pred).numpy()
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel('predicted')
plt.ylabel('actual')
plt.title('confusion matrix (test data)')
plt.show()

> **[교사용]** **성취수준 연결 [12인기02-06]**: 테스트 데이터로 평가해 수치를 기록하면 C 수준의 '성능 평가'를 충족합니다. 훈련 정확도와 테스트 정확도의 차이를 해석하고, 혼동 행렬에서 자주 헷갈린 숫자 쌍(예: 4와 9, 3과 5, 7과 2)과 그 이유를 서술하면 A·B 수준의 근거가 됩니다.
> 혼동 행렬은 행이 정답, 열이 예측입니다. 대각선은 맞힌 개수입니다.

In [ ]:
#@title ⑥ [단계 4] 예측하기: 신경망이 틀린 숫자 보기
wrong = np.where(pred != y_test)[0]
print(f'테스트 데이터 {len(y_test):,}개 중 틀린 개수: {len(wrong)}개')
print('그림 위의 표시는 "정답 -> 신경망의 답"입니다.')
plt.figure(figsize=(12, 2.8))
for k, idx in enumerate(wrong[:12]):
    plt.subplot(1, 12, k + 1)
    plt.imshow(X_test[idx], cmap='gray')
    plt.title(f'{y_test[idx]} -> {pred[idx]}', fontsize=9)
    plt.axis('off')
plt.show()

> **[교사용]** 틀린 숫자 중에는 사람이 봐도 애매한 것이 많습니다. "신경망이 틀린 걸까, 데이터가 애매한 걸까?"를 토론하면 교과서의 'AI 속이기' 활동과 이어집니다.

## 연구 노트에 기록하기

교과서 페이지의 연구 노트나 선생님이 안내한 구글 폼에 아래 표를 채워 제출합니다.

| 시도 | 은닉층 1 | 은닉층 2 | 활성화 함수 | 학습 횟수 | 훈련 정확도 | 테스트 정확도 |
|---|---|---|---|---|---|---|
| 기본 | 128 | 64 | relu | 10 | | |
| 1 | | | | | | |
| 2 | | | | | | |

- 내 예측과 실제 결과는 얼마나 달랐나요?
- 혼동 행렬에서 신경망이 가장 자주 헷갈린 숫자 쌍은 무엇인가요?

## 도전: 코드 한 줄 고치기
아래 칸은 코드가 펼쳐져 있습니다. `# 여기에 한 줄 추가` 자리에 은닉층을 하나 더 넣어 보세요. 위에 있는 `layers.Dense(...)` 줄을 참고하면 됩니다. 층을 넣기 전과 후의 테스트 정확도를 비교하세요.

In [ ]:
challenge = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    # 여기에 한 줄 추가
    layers.Dense(10, activation='softmax'),
])
challenge.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
challenge.fit(X_train, y_train, epochs=5, validation_split=0.1, batch_size=128, verbose=2)
print('테스트 정확도:', round(challenge.evaluate(X_test, y_test, verbose=0)[1] * 100, 2), '%')

> **[교사용]** **정답 예**: `layers.Dense(64, activation='relu'),`  (쉼표를 빠뜨려 오류가 나는 경우가 많습니다.)
> 층을 하나 더 넣어도 정확도가 크게 오르지 않을 수 있습니다. "층이 많다고 항상 좋은 것은 아니다"를 1차시 은닉층 실험실과 연결해 정리합니다.